# RFM + Clustering + Churn (v2)

Este notebook segue a mesma lógica da versão original:

**Dados transacionais → tabela de clientes → RFM + outras features → clustering → segmentação →
modelo de churn → probabilidade → priorização → ação de retenção.**

Em relação à v1, só 3 coisas mudaram (o resto do notebook é o mesmo):

1. **O churn deixou de ser uma fórmula das próprias features.** Antes, `churn` era calculado
   diretamente a partir de `recency`/`frequency`/`monetary` — ou seja, o alvo já "continha" a
   resposta que o modelo deveria descobrir. Agora ele vem do segmento latente do cliente (que
   gerou as compras, mas não é uma feature do modelo) mais uma dose de aleatoriedade — do jeito
   que churn realmente funciona: correlacionado com o comportamento observado, mas não uma
   função determinística dele.
2. **O k do K-Means passou a ser escolhido por silhouette score**, em vez de fixado em 4 sem
   justificativa.
3. **A avaliação do modelo usa validação cruzada (5 folds)** em vez de um único split
   aleatório, para não tirar conclusão de um resultado que pode ser sorte/azar de split.

> Os dados continuam sintéticos, para fins didáticos.

## 1. Visão do projeto

1. Gerar/ler dados transacionais
2. Construir uma tabela de clientes
3. Calcular RFM
4. Criar outras variáveis comportamentais
5. Fazer EDA
6. Aplicar K-Means para segmentação (k escolhido por silhouette)
7. Interpretar os clusters
8. Definir churn (a partir do segmento latente, não das features observáveis)
9. Treinar modelo supervisionado
10. Comparar modelo com e sem cluster, usando validação cruzada
11. Gerar `P(churn)` para a base
12. Priorizar clientes considerando risco e valor

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

np.random.seed(42)

pd.set_option("display.max_columns", 50)

## 2. Dados transacionais

Em um projeto real, esta tabela poderia vir de SQL.

Exemplo conceitual:

```sql
SELECT
    customer_id,
    purchase_date,
    amount,
    product_category,
    channel
FROM transactions;
```

Aqui vamos gerar uma base sintética — exatamente como na v1. Cada cliente tem um **segmento
latente** (`high_value`, `regular`, `low_value`, `at_risk`) que determina quantas compras faz e
o quão recentes elas são. Esse segmento nunca é usado como feature do modelo — ele só existe
para gerar os dados, do mesmo jeito que, no mundo real, existe um comportamento verdadeiro do
cliente que a gente não observa diretamente, só através de RFM.

In [ ]:
# Gerando clientes
n_customers = 2500
customer_ids = np.arange(1, n_customers + 1)

# Perfil latente usado apenas para gerar dados sintéticos
segment = np.random.choice(
    ["high_value", "regular", "low_value", "at_risk"],
    size=n_customers,
    p=[0.15, 0.45, 0.30, 0.10]
)

rows = []

categories = ["Skin Care", "Filler", "Biostimulator", "Toxin"]

for cid, seg in zip(customer_ids, segment):
    if seg == "high_value":
        n_orders = np.random.poisson(18) + 2
        avg_amount = 900
        recency_max = 45
    elif seg == "regular":
        n_orders = np.random.poisson(8) + 1
        avg_amount = 500
        recency_max = 90
    elif seg == "low_value":
        n_orders = np.random.poisson(3) + 1
        avg_amount = 220
        recency_max = 180
    else:
        n_orders = np.random.poisson(5) + 1
        avg_amount = 400
        recency_max = 300

    # Compras relativamente recentes ou antigas, dependendo do perfil
    days_ago = np.random.randint(1, recency_max + 1, size=n_orders)

    for d in days_ago:
        amount = max(30, np.random.normal(avg_amount, avg_amount * 0.25))
        rows.append({
            "customer_id": cid,
            "purchase_date": pd.Timestamp("2026-08-31") - pd.Timedelta(days=int(d)),
            "amount": amount,
            "product_category": np.random.choice(categories),
            "channel": np.random.choice(["Online", "Sales Rep", "Distributor"])
        })

transactions = pd.DataFrame(rows)

transactions.head(), transactions.shape

## 3. Tabela de clientes

A partir das transações, transformamos várias linhas por cliente em uma linha por cliente.

Esta é a **tabela analítica** que será usada nas etapas seguintes. Guardamos também o
`segment` latente à parte (`customer_segment`) — ele só será usado na Seção 8 para gerar o
rótulo de churn, nunca como feature de entrada do modelo.

In [ ]:
reference_date = pd.Timestamp("2026-08-31")

customer = (
    transactions.groupby("customer_id")
    .agg(
        last_purchase=("purchase_date", "max"),
        frequency=("purchase_date", "count"),
        monetary=("amount", "sum"),
        avg_ticket=("amount", "mean"),
        first_purchase=("purchase_date", "min")
    )
    .reset_index()
)

customer["recency"] = (
    reference_date - customer["last_purchase"]
).dt.days

customer["tenure_days"] = (
    reference_date - customer["first_purchase"]
).dt.days

customer["avg_days_between_orders"] = (
    customer["tenure_days"] / customer["frequency"].clip(lower=2)
)

# segmento latente por cliente, guardado à parte (não é feature do modelo)
customer_segment = pd.Series(segment, index=customer_ids, name="segment")

customer.head()

## 4. RFM

Agora temos:

- **Recency:** dias desde a última compra
- **Frequency:** número de compras
- **Monetary:** valor total comprado

Essas três variáveis formam o RFM.

In [ ]:
rfm = customer[[
    "customer_id", "recency", "frequency", "monetary"
]].copy()

rfm.describe()

## 5. EDA

Antes de modelar, investigamos distribuição, valores extremos e relações entre as variáveis.

Em dados reais, esta etapa também incluiria:

- valores ausentes
- duplicidades
- inconsistências
- outliers
- regras de negócio
- distribuição temporal
- estabilidade dos dados

In [ ]:
fig, ax = plt.subplots()
ax.hist(customer["recency"], bins=40)
ax.set_title("Distribuição de Recency")
ax.set_xlabel("Dias desde a última compra")
ax.set_ylabel("Clientes")
plt.show()

fig, ax = plt.subplots()
ax.scatter(customer["frequency"], customer["monetary"], alpha=0.25)
ax.set_title("Frequency vs Monetary")
ax.set_xlabel("Frequency")
ax.set_ylabel("Monetary")
plt.show()

## 6. Clustering

O K-Means é **não supervisionado**: não usamos churn para criar os grupos.

Usaremos RFM + algumas features comportamentais. Como as escalas são diferentes, primeiro
padronizamos as variáveis.

**Diferença para a v1:** ali `n_clusters=4` era escolhido sem nenhuma validação. Aqui testamos
k de 2 a 8 e escolhemos o k com maior silhouette score — uma medida de quão bem separados e
coesos os clusters ficam.

In [ ]:
cluster_features = [
    "recency",
    "frequency",
    "monetary",
    "avg_ticket",
    "tenure_days"
]

X_cluster = customer[cluster_features].copy()

scaler_cluster = StandardScaler()
X_cluster_scaled = scaler_cluster.fit_transform(X_cluster)

silhouette_by_k = {}
for k in range(2, 9):
    labels = KMeans(n_clusters=k, random_state=42, n_init=20).fit_predict(X_cluster_scaled)
    silhouette_by_k[k] = silhouette_score(X_cluster_scaled, labels)

best_k = max(silhouette_by_k, key=silhouette_by_k.get)
print("Silhouette por k:", {k: round(v, 3) for k, v in silhouette_by_k.items()})
print("k escolhido:", best_k)

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=20
)

customer["cluster"] = kmeans.fit_predict(X_cluster_scaled)

customer.groupby("cluster")[cluster_features].mean().round(2)

## 7. Interpretando os clusters

Os números `0, 1, 2, ...` não significam nada por si só.

Precisamos olhar para as médias e dar uma interpretação de negócio.

Uma forma simples é criar um resumo ordenado por Recency, Frequency e Monetary.

In [ ]:
cluster_summary = (
    customer.groupby("cluster")[cluster_features]
    .agg(["mean", "median"])
    .round(2)
)

cluster_summary

In [ ]:
# Perfil simplificado de cada cluster
profile = (
    customer.groupby("cluster")
    .agg(
        customers=("customer_id", "count"),
        recency=("recency", "mean"),
        frequency=("frequency", "mean"),
        monetary=("monetary", "mean"),
        avg_ticket=("avg_ticket", "mean"),
        tenure_days=("tenure_days", "mean")
    )
    .round(2)
)

profile

## 8. Criando o target de churn

Agora começa o problema **supervisionado**.

**Diferença para a v1:** ali, `churn` era calculado com uma fórmula de `recency`, `frequency` e
`monetary` — as mesmas variáveis que depois entravam no modelo como preditoras. Isso é
vazamento: o alvo já era, por construção, quase uma transformação das features, então qualquer
modelo "acertava" trivialmente e a métrica não dizia nada sobre capacidade preditiva real.

Aqui o churn vem do **segmento latente** do cliente (a mesma variável que gerou as compras lá
na Seção 2), com uma probabilidade de churn por segmento. Isso é mais realista: no mundo real,
quem decide se o cliente vai comprar de novo é o comportamento/intenção verdadeira dele — RFM é
só um **sintoma observável** desse comportamento, correlacionado mas não idêntico a ele. O
segmento nunca entra como feature do modelo (só o usamos aqui, para gerar o rótulo).

In [ ]:
# Probabilidade de churn por segmento latente — não é função de recency/frequency/monetary.
# É o "gerador" verdadeiro dos dados, do mesmo jeito que o segmento gerou as compras na Seção 2.
segment_churn_rate = {
    "high_value": 0.05,
    "regular": 0.15,
    "low_value": 0.35,
    "at_risk": 0.70,
}

churn_prob = customer_segment.reindex(customer["customer_id"]).map(segment_churn_rate).values
customer["churn"] = np.random.binomial(1, churn_prob)

customer["churn"].mean()

## 9. Preparando as features

Agora temos duas possibilidades:

### Modelo A — sem cluster

Usamos apenas as características do cliente.

### Modelo B — com cluster

Usamos as mesmas características + o segmento encontrado pelo K-Means.

Isso permite testar empiricamente se a segmentação adiciona informação para a previsão de churn.

In [ ]:
features_base = [
    "recency",
    "frequency",
    "monetary",
    "avg_ticket",
    "tenure_days"
]

features_cluster = features_base + ["cluster"]

target = "churn"

X_base = customer[features_base]
X_cluster_feat = customer[features_cluster]
y = customer[target]

## 10. Modelo supervisionado — sem cluster vs. com cluster

Mesma pergunta da v1:

> Quanto conseguimos prever churn com as variáveis comportamentais — e o cluster ajuda além
> disso?

**Diferença para a v1:** ali, os dois modelos eram avaliados com um único split aleatório
treino/teste — um resultado que pode ser só sorte (ou azar) daquele split específico. Aqui
usamos validação cruzada estratificada (5 folds): cada modelo é treinado e avaliado 5 vezes, em
partições diferentes, e reportamos a média e o desvio-padrão do ROC-AUC e do PR-AUC.

In [ ]:
model_base = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])

preprocessor_cluster = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), features_base),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), ["cluster"]),
])

model_cluster = Pipeline([
    ("preprocessor", preprocessor_cluster),
    ("model", LogisticRegression(max_iter=2000))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def cv_summary(model, X, y, cv):
    roc_auc = cross_val_score(model, X, y, cv=cv, scoring="roc_auc")
    pr_auc = cross_val_score(model, X, y, cv=cv, scoring="average_precision")
    return {
        "ROC_AUC_mean": roc_auc.mean(),
        "ROC_AUC_std": roc_auc.std(),
        "PR_AUC_mean": pr_auc.mean(),
        "PR_AUC_std": pr_auc.std(),
    }

comparison = pd.DataFrame({
    "Sem cluster": cv_summary(model_base, X_base, y, cv),
    "Com cluster": cv_summary(model_cluster, X_cluster_feat, y, cv),
}).round(3)

comparison

## 11. Comparação

O ponto não é assumir que o cluster melhora o modelo — é o que a tabela acima testa. Se o
`ROC_AUC_mean` do modelo com cluster for maior que o do modelo sem cluster **por uma margem
maior que o desvio-padrão** dos dois, há indício de que o cluster ajuda. Se as médias forem
próximas e os desvios se sobrepõem, o cluster provavelmente não está agregando informação além
do que RFM já entrega — e é um resultado válido, não um fracasso do notebook.

## 12. Gerando probabilidade de churn para a base

Depois de comparar os dois modelos, treinamos o escolhido (aqui, o modelo com cluster) em toda
a base e usamos para gerar `P(churn)` de cada cliente. É esse modelo final, treinado com todos
os dados disponíveis, que seria usado na prática — a validação cruzada da Seção 10 é só para
estimar o quão bom ele deve ser, não o modelo em si.

In [ ]:
model_cluster.fit(X_cluster_feat, y)
customer["p_churn"] = model_cluster.predict_proba(X_cluster_feat)[:, 1]

customer[[
    "customer_id",
    "cluster",
    "monetary",
    "p_churn"
]].sort_values("p_churn", ascending=False).head(10)

## 13. Priorização de clientes

Probabilidade de churn não é a mesma coisa que prioridade comercial.

Um cliente com:

- 90% de churn
- R$ 100 de valor

pode ser menos prioritário que outro com:

- 65% de churn
- R$ 10.000 de valor

Por isso podemos combinar risco e valor.

In [ ]:
customer["expected_value_at_risk"] = (
    customer["p_churn"] * customer["monetary"]
)

priority = customer[[
    "customer_id",
    "cluster",
    "recency",
    "frequency",
    "monetary",
    "p_churn",
    "expected_value_at_risk"
]].sort_values(
    "expected_value_at_risk",
    ascending=False
)

priority.head(20)

## 14. Transformando o modelo em ação

Uma regra simples: cruzar risco (`p_churn`) e valor (`monetary`) usando a mediana de cada um
como corte.

- **Alto risco + alto valor:** ação prioritária de retenção
- **Alto risco + baixo valor:** ação automatizada/de baixo custo
- **Baixo risco + alto valor:** relacionamento/fidelização
- **Baixo risco + baixo valor:** comunicação padrão

O modelo não decide sozinho qual campanha executar. Ele produz uma informação para apoiar a
decisão. Na v1 essa régua só existia como texto — aqui ela é implementada.

In [ ]:
risk_median = customer["p_churn"].median()
value_median = customer["monetary"].median()

def action(row):
    high_risk = row["p_churn"] >= risk_median
    high_value = row["monetary"] >= value_median
    if high_risk and high_value:
        return "Retenção prioritária"
    if high_risk and not high_value:
        return "Ação automatizada de baixo custo"
    if not high_risk and high_value:
        return "Relacionamento / fidelização"
    return "Comunicação padrão"

customer["acao_recomendada"] = customer.apply(action, axis=1)
customer["acao_recomendada"].value_counts()

## 15. O que mudou da v1 para a v2

| Problema na v1 | Correção na v2 |
|---|---|
| `churn` era função determinística de `recency`/`frequency`/`monetary` | `churn` vem do segmento latente + aleatoriedade — correlacionado com RFM, não calculado a partir dele |
| `n_clusters=4` fixo, sem validação | k escolhido por silhouette score |
| Um único split aleatório para comparar os modelos | Validação cruzada (5 folds), reportando média ± desvio-padrão |

O resto do notebook — RFM, clustering, cluster como feature, comparação empírica dos modelos,
`P(churn)` × valor → priorização → ação — é a mesma lógica da v1.

### Limitações que continuam

- Os dados são sintéticos; o objetivo é ilustrar o método, não gerar conclusões de negócio reais.
- Em produção, o ideal é validar o modelo também **fora do tempo** (treinar num período,
  testar num período posterior) — isso exige múltiplos cortes históricos, o que está fora do
  escopo deste exemplo com um único snapshot de dados.
- A definição de churn (aqui, uma probabilidade por segmento) precisa, num projeto real, ser
  substituída pela definição de negócio de fato (ex.: X dias sem comprar).